# Model Interpretability

This notebook explores the **interpretability vs accuracy tradeoff** by comparing:
- **Linear Regression** -- inherently interpretable (coefficients directly explain feature effects)
- **Random Forest** -- a black-box model that may capture non-linear relationships

We use the **California Housing dataset** and investigate models with:
1. Accuracy comparison (RMSE, R-squared)
2. **Partial Dependence Plots (PDP)** -- how does a feature affect predictions on average?
3. **Individual Conditional Expectation (ICE)** plots -- how does a feature affect individual predictions?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import PartialDependenceDisplay

sns.set_style('whitegrid')
np.random.seed(42)

## 1. Load and Prepare Data

In [ ]:
housing = fetch_california_housing()
X, y = housing.data, housing.target
feature_names = housing.feature_names

print(f'Dataset shape: {X.shape}')
print(f'Target: Median house value (in $100k)')
print(f'Features: {feature_names}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale for linear regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nTrain: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 2. Train Models and Compare Accuracy

In [ ]:
# Linear Regression (interpretable)
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# Random Forest (black box)
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Compare
results = {
    'Linear Regression': {
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        'R2': r2_score(y_test, y_pred_lr)
    },
    'Random Forest': {
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        'R2': r2_score(y_test, y_pred_rf)
    }
}

print(f'{"Model":<25} {"RMSE":<10} {"R-squared":<10}')
print('-' * 45)
for name, metrics in results.items():
    print(f'{name:<25} {metrics["RMSE"]:<10.4f} {metrics["R2"]:<10.4f}')

In [ ]:
# Visual comparison of predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (name, y_pred) in zip(axes, [('Linear Regression', y_pred_lr), ('Random Forest', y_pred_rf)]):
    ax.scatter(y_test, y_pred, alpha=0.3, s=10, color='#3498db')
    ax.plot([0, 5], [0, 5], 'r--', linewidth=2, label='Perfect prediction')
    ax.set_xlabel('Actual', fontsize=12)
    ax.set_ylabel('Predicted', fontsize=12)
    ax.set_title(f'{name}\nRMSE={results[name]["RMSE"]:.3f}, R2={results[name]["R2"]:.3f}',
                 fontsize=13)
    ax.legend()
    ax.set_xlim(0, 5.2)
    ax.set_ylim(0, 5.2)

plt.tight_layout()
plt.show()

## 3. Linear Regression: Inherent Interpretability

Linear regression coefficients directly tell us the effect of each feature (after scaling).

In [ ]:
coef_sorted_idx = np.argsort(np.abs(lr.coef_))

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in lr.coef_[coef_sorted_idx]]
ax.barh(range(len(feature_names)), lr.coef_[coef_sorted_idx], color=colors)
ax.set_yticks(range(len(feature_names)))
ax.set_yticklabels(np.array(feature_names)[coef_sorted_idx])
ax.set_xlabel('Coefficient (standardized features)', fontsize=12)
ax.set_title('Linear Regression Coefficients\n(Green = positive effect, Red = negative effect)',
             fontsize=13)
ax.axvline(x=0, color='black', linewidth=1)
plt.tight_layout()
plt.show()

print('Interpretation:')
print(f'  Intercept: {lr.intercept_:.4f}')
for i in coef_sorted_idx[::-1]:
    direction = 'increases' if lr.coef_[i] > 0 else 'decreases'
    print(f'  {feature_names[i]}: 1 std increase {direction} house value by ${abs(lr.coef_[i]) * 100000:.0f}')

## 4. Partial Dependence Plots (PDP)

PDPs show how the **average prediction** changes as we vary one feature while keeping all others fixed. This works for any model.

In [ ]:
# Select the most important features for PDP
important_features = ['MedInc', 'AveRooms', 'HouseAge', 'Latitude']
feature_indices = [list(feature_names).index(f) for f in important_features]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Linear Regression PDPs
PartialDependenceDisplay.from_estimator(
    lr, X_train_scaled, features=feature_indices,
    feature_names=feature_names, ax=axes[0],
    kind='average', grid_resolution=50
)
axes[0, 0].set_ylabel('Linear Regression\nPartial Dependence', fontsize=11)

# Random Forest PDPs
PartialDependenceDisplay.from_estimator(
    rf, X_train, features=feature_indices,
    feature_names=feature_names, ax=axes[1],
    kind='average', grid_resolution=50
)
axes[1, 0].set_ylabel('Random Forest\nPartial Dependence', fontsize=11)

fig.suptitle('Partial Dependence Plots: Linear vs Random Forest', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 5. ICE (Individual Conditional Expectation) Plots

While PDPs show the **average** effect, ICE plots show how each **individual sample's** prediction changes with a feature. This can reveal heterogeneous effects that PDPs hide.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ICE plots for Random Forest (more interesting than linear)
for ax, feat_idx, feat_name in zip(axes.flat, feature_indices, important_features):
    PartialDependenceDisplay.from_estimator(
        rf, X_train[:500], features=[feat_idx],  # subsample for speed
        feature_names=feature_names, ax=ax,
        kind='both',  # both ICE lines and PDP
        ice_lines_kw={'color': '#3498db', 'alpha': 0.05, 'linewidth': 0.5},
        pd_line_kw={'color': '#e74c3c', 'linewidth': 3},
        grid_resolution=50
    )
    ax.set_title(f'ICE + PDP: {feat_name} (Random Forest)', fontsize=12)

plt.tight_layout()
plt.show()

## 6. Interpretability vs Accuracy Tradeoff

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

models = {
    'Linear Regression': {'accuracy': results['Linear Regression']['R2'], 'interpretability': 9},
    'Random Forest': {'accuracy': results['Random Forest']['R2'], 'interpretability': 4},
}

for name, vals in models.items():
    ax.scatter(vals['interpretability'], vals['accuracy'], s=200, zorder=5)
    ax.annotate(f'{name}\n(R2={vals["accuracy"]:.3f})', 
                (vals['interpretability'], vals['accuracy']),
                textcoords='offset points', xytext=(10, 10),
                fontsize=11, fontweight='bold')

ax.set_xlabel('Interpretability Score (higher = more interpretable)', fontsize=12)
ax.set_ylabel('Accuracy (R-squared)', fontsize=12)
ax.set_title('Interpretability vs Accuracy Tradeoff', fontsize=14)
ax.set_xlim(0, 10)
ax.set_ylim(0, 1)

# Add arrow showing the tradeoff
ax.annotate('', xy=(2, 0.9), xytext=(8, 0.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2, ls='--'))
ax.text(5, 0.65, 'Typical\nTradeoff', ha='center', va='center',
        fontsize=10, color='gray', style='italic')

plt.tight_layout()
plt.show()

## Summary

### Key Takeaways

| Aspect | Linear Regression | Random Forest |
|--------|------------------|---------------|
| **Interpretability** | High -- coefficients are directly meaningful | Low -- complex ensemble of trees |
| **Accuracy** | Moderate -- assumes linear relationships | Higher -- captures non-linearity |
| **PDP shape** | Always linear (by construction) | Can show non-linear, threshold effects |
| **ICE plots** | All lines parallel (same effect for all samples) | Lines can cross (heterogeneous effects) |

### When to use what
- **High-stakes decisions** (medical, legal): prefer interpretable models or supplement black-box models with strong XAI methods
- **Pure prediction tasks**: use the most accurate model and explain it post-hoc with PDPs, ICE plots, or SHAP
- **Feature understanding**: PDPs and ICE plots are model-agnostic tools that work with any model

### PDP vs ICE
- PDPs can be misleading when features interact -- they show the **average** which may not represent any individual
- ICE plots reveal **heterogeneous effects** -- when different subgroups respond differently to a feature
- Always plot ICE alongside PDP to check for hidden interactions